In [2]:
import pandas as pd

# Ladda in CSV dataset
netflix = pd.read_csv("netflix Dataset.csv")

netflix.head()

print(netflix.dtypes)

Show_Id         object
Category        object
Title           object
Director        object
Cast            object
Country         object
Release_Date    object
Rating          object
Duration        object
Type            object
Description     object
dtype: object


In [6]:
netflix['Release_Year'] = pd.to_datetime(netflix['Release_Date'], errors='coerce').dt.year
netflix['Release_Year'] = netflix['Release_Year'].fillna(0).astype(int)
def parse_duration(x):
    if pd.isna(x):
        return 0
    if "Season" in x:   # TV Show
        return int(x.split()[0])   # antal säsonger
    if "min" in x:      # Movie
        return int(x.split()[0])   # antal minuter
    return 0

netflix['Duration_num'] = netflix['Duration'].apply(parse_duration)

print(netflix[['Title', 'Duration', 'Duration_num', 'Release_Date', 'Release_Year']].head(10))
print(netflix.dtypes)

   Title   Duration  Duration_num       Release_Date  Release_Year
0     3%  4 Seasons             4    August 14, 2020          2020
1  07:19     93 min            93  December 23, 2016          2016
2  23:59     78 min            78  December 20, 2018          2018
3      9     80 min            80  November 16, 2017          2017
4     21    123 min           123    January 1, 2020          2020
5     46   1 Season             1       July 1, 2017          2017
6    122     95 min            95       June 1, 2020          2020
7    187    119 min           119   November 1, 2019          2019
8    706    118 min           118      April 1, 2019          2019
9   1920    143 min           143  December 15, 2017          2017
Show_Id         object
Category        object
Title           object
Director        object
Cast            object
Country         object
Release_Date    object
Rating          object
Duration        object
Type            object
Description     object
Release_Ye

In [33]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.manifold import TSNE

from scipy.sparse import hstack
import matplotlib.pyplot as plt

# Steg 1: Läs in data
df = pd.read_csv("Netflix Dataset.csv")

# Separera filmer (med "min") och serier (med "Season")
movies = df[df['Duration'].str.contains("min", na=False)].copy()
shows = df[df['Duration'].str.contains("Season", na=False)].copy()

# Extrahera minuter för filmer
movies['Minutes'] = movies['Duration'].str.extract(r'(\d+)').astype(float)

# Extrahera antal säsonger för serier
shows['Seasons'] = shows['Duration'].str.extract(r'(\d+)').astype(float)

# Ta bort siffror ur "Duration" (behåll bara texten "min"/"Season")
movies['Duration'] = movies['Duration'].str.replace(r'\d+', '', regex=True).str.strip()
shows['Duration'] = shows['Duration'].str.replace(r'\d+', '', regex=True).str.strip()


# Kolla resultat
print(f"Antal filmer: {movies.shape[0]}")
print(movies[['Title', 'Duration', 'Minutes']].head())

print(f"Antal serier: {shows.shape[0]}")
print(shows[['Title', 'Duration', 'Seasons']].head())


Antal filmer: 5379
   Title Duration  Minutes
1  07:19      min     93.0
2  23:59      min     78.0
3      9      min     80.0
4     21      min    123.0
6    122      min     95.0
Antal serier: 2410
     Title Duration  Seasons
0       3%  Seasons      4.0
5       46   Season      1.0
11    1983   Season      1.0
12    1994   Season      1.0
16  Feb-09   Season      1.0


In [ ]:
# Hantera saknad description
movies['Description'] = movies['Description'].fillna("")

# Extrahera år
movies['Release_Year'] = pd.to_datetime(movies['Release_Date'], errors='coerce').dt.year.fillna(0).astype(int)

# Steg 2: generera features
# Text (beskrivning)
tfidf = TfidfVectorizer(stop_words="english", max_features=3000)
tfidf_desc = tfidf.fit_transform(movies['Description'].fillna(""))

# Text-feature från description
tfidf = TfidfVectorizer(stop_words="english", max_features=3000)
tfidf_desc = tfidf.fit_transform(movies['Description'])

# Kategoriska features: Type, Category, Rating, Country
cat_cols = ['Type', 'Category', 'Rating', 'Country']
encoder = OneHotEncoder(handle_unknown='ignore')
cat_feats = encoder.fit_transform(movies[cat_cols].fillna("Unknown"))

# Numeriska features: år, minuter
num_feats = movies[['Release_Year', 'Minutes']].fillna(0).values
scaler = StandardScaler()
num_feats_scaled = scaler.fit_transform(num_feats)

# Kombinera alla features
# Väga olika delar
tfidf_w = 0.7
cat_w   = 1.2
num_w   = 0.5

X = hstack([
    tfidf_desc * tfidf_w,
    cat_feats * cat_w,
    num_feats_scaled * num_w
])

# Säkerställ att X är CSR (snabbare och indexerbar)
from scipy.sparse import csr_matrix
X_csr = csr_matrix(X)

# Steg 3: klustring
kmeans = KMeans(n_clusters=6, random_state=42)
clusters = kmeans.fit_predict(X_csr)

# Lägg till kluster i movies DataFrame
movies['Cluster'] = clusters

# Title till idx
title_to_idx = pd.Series(movies.index.values, index=movies['Title']).to_dict()

# Hjälpfunktion: titel → filmer i samma kluster
def get_similar_cluster(title, top_n=5):
    if title not in title_to_idx:
        return []

    idx = title_to_idx[title]
    pos = movies.index.get_loc(idx)        # position i DataFrame
    cluster_id = movies.loc[idx, 'Cluster']
    
    # Hämta filmer i samma kluster
    movie_type = movies.loc[idx, 'Type']
    cluster_indices = movies[(movies['Cluster'] == cluster_id) & (movies['Type'] == movie_type)].index
    cluster_indices = cluster_indices[cluster_indices != idx] 


    
    all_candidates = list(cluster_indices)  # börja med samma kluster
    
    # Om för få, leta i närmaste kluster
    if len(all_candidates) < top_n:
        # Avstånd mellan klustercentroider
        centroid_distances = euclidean_distances(
            kmeans.cluster_centers_[cluster_id].reshape(1, -1),
            kmeans.cluster_centers_
        ).flatten()
        
        # sortera närmsta kluster (exkludera eget)
        nearest_clusters = np.argsort(centroid_distances)[1:]
        
        for nc in nearest_clusters:
            more_candidates = movies[movies['Cluster'] == nc].index.tolist()
            all_candidates.extend(more_candidates)
            if len(all_candidates) >= top_n:
                break
    # Beräkna likhet inom kandidaterna
    candidate_positions = [movies.index.get_loc(i) for i in all_candidates]
    candidate_X = X_csr[candidate_positions]
    movie_vec = X_csr[pos]
    
    sims = cosine_similarity(movie_vec, candidate_X).flatten()
    top_positions = np.argsort(sims)[::-1][:top_n]
    
    similar_idx = [all_candidates[i] for i in top_positions]
    return movies.loc[similar_idx, 'Title'].tolist(), similar_idx

# --- Visualization ---
def plot_cluster_with_neighbors_tsne(title, top_n=5):
    idx = title_to_idx[title]
    pos = movies.index.get_loc(idx)
    
    titles, sim_idx = get_similar_cluster(title, top_n)
    sim_positions = [movies.index.get_loc(i) for i in sim_idx]
    
    # Endast klustret + 50 närmaste grannar
    subset_idx = [pos] + sim_positions
    X_subset = X_csr[subset_idx].toarray()
    X_subset_50d = PCA(n_components=50, random_state=42).fit_transform(X_subset)
    X_2d = TSNE(n_components=2, perplexity=5, random_state=42).fit_transform(X_subset_50d)

    
    plt.figure(figsize=(11, 8))
    
    # plotta varje kluster med egen färg + label
    for cluster_id, group in movies.groupby("Cluster"):
        cluster_pos = [movies.index.get_loc(i) for i in group.index]
        label = f"{cluster_id}: {group['Type'].mode()[0]}"
        plt.scatter(
            X_2d[cluster_pos,0],
            X_2d[cluster_pos,1],
            label=label,
            alpha=0.4
        )

    
    # markera vald film (röd)
    plt.scatter(X_2d[pos,0], X_2d[pos,1], c="red", s=200, label=title, edgecolor="black", marker="*")
    
    # markera grannar (blå)
    plt.scatter(X_2d[sim_positions,0], X_2d[sim_positions,1], c="blue", s=120, label="Neighbors", edgecolor="black")
    
    plt.legend()
    plt.title(f"t-SNE Cluster Visualization för '{title}' och dess {top_n} närmaste")
    plt.show()
    
print(f"\nDe {top_n} närmsta filmerna till '{title}':")
for i, t in enumerate(titles, 1):
    print(f"{i}. {t}")

plot_cluster_with_neighbors_tsne("Inception", top_n=5)

print("\n--- Klusterprofilering ---")
for cluster_id, group in movies.groupby("Cluster"):
    most_common_type = group['Type'].mode()[0] if not group['Type'].mode().empty else "Okänd"
    most_common_cat = group['Category'].mode()[0] if not group['Category'].mode().empty else "Okänd"
    most_common_rating = group['Rating'].mode()[0] if not group['Rating'].mode().empty else "Okänd"
    
    print(f"Cluster {cluster_id}:")
    print(f"   Vanligaste Type     = {most_common_type}")
    print(f"   Vanligaste Category = {most_common_cat}")
    print(f"   Vanligaste Rating   = {most_common_rating}")
    print(f"   Antal titlar        = {len(group)}\n")



# Testa med Inception
titles, sim_idx = get_similar_cluster("Inception", top_n=5)

print(f"\nDe 5 närmsta filmerna till 'Inception':")
for i, t in enumerate(titles, 1):
    print(f"{i}. {t}")


KeyboardInterrupt: 